# Phase 2 — GRU Model Architecture & Data Pipeline
## Container Resource Prediction | PyTorch

---

### What this notebook contains

| Component | Description |
|---|---|
| `SequenceDataset` | Custom Dataset wrapping (X, y) NumPy arrays |
| `load_dataset()` | Loads .npy files for a given horizon + split |
| `create_dataloader()` | Configures a DataLoader from arrays |
| `build_dataloaders()` | Builds train / val / test loaders in one call |
| `initialize_weights()` | Orthogonal (GRU) + Xavier (Linear) init |
| `GRUModel` | 2-layer GRU -> Dropout -> FC -> ReLU -> Output |

**Does NOT include:** training, evaluation, Optuna, plotting, or saving models.

---
**Author:** Team-Dracasys | **Phase:** 2 of 4

## Step 1: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted.")

In [ ]:
from pathlib import Path

# Sequences produced by Phase 1 pipeline
sequences_path = Path('/content/processed_data/sequences')

print(f"Sequences path : {sequences_path}")
print(f"Path exists    : {sequences_path.exists()}")

if sequences_path.exists():
    npy_files = sorted(sequences_path.glob('*.npy'))
    print(f"Total .npy files: {len(npy_files)}")
    for f in npy_files[:6]:
        print(f"  {f.name}")
    if len(npy_files) > 6:
        print(f"  ... and {len(npy_files)-6} more")

## Step 2: Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from typing import Dict, Tuple
import logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
print(f"CUDA available  : {torch.cuda.is_available()}")

## Step 3: SequenceDataset

A custom `Dataset` that:
- Accepts raw NumPy arrays `(X, y)` from the Phase 1 `.npy` files
- Converts them to `float32` tensors **once** in `__init__` (not per-sample)
- Exposes `__len__` and `__getitem__` so PyTorch's DataLoader can index into it

In [ ]:
class SequenceDataset(Dataset):
    """
    PyTorch Dataset for pre-computed sliding-window sequences.

    Parameters
    ----------
    X : np.ndarray  shape (n_samples, seq_len, n_features)
        Input windows produced by the Phase 1 sequence generator.
        Example: (174220, 240, 27)
    y : np.ndarray  shape (n_samples, n_targets)
        Regression targets at the chosen prediction horizon.
        Example: (174220, 4)

    Notes
    -----
    Arrays are cast to float32 ONCE in __init__.
    No normalisation is applied here — sequences are already
    z-score normalised from Phase 1.
    """

    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        if len(X) != len(y):
            raise ValueError(
                f"X and y must have the same number of samples "
                f"(got X={len(X)}, y={len(y)})"
            )
        # np.ascontiguousarray ensures tensors are in contiguous memory
        # which is required for efficient GPU transfer via pin_memory
        self.X = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))

    def __len__(self) -> int:
        """Total number of samples in the dataset."""
        return len(self.X)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Return one (input_window, target) pair.

        Returns
        -------
        x : torch.Tensor  shape (seq_len, n_features)  e.g. (240, 27)
        y : torch.Tensor  shape (n_targets,)            e.g. (4,)
        """
        return self.X[idx], self.y[idx]

    # ── Informational properties ──────────────────────────────────────
    @property
    def n_samples(self)  -> int: return len(self.X)
    @property
    def seq_len(self)    -> int: return self.X.shape[1]
    @property
    def n_features(self) -> int: return self.X.shape[2]
    @property
    def n_targets(self)  -> int: return self.y.shape[1]

    def __repr__(self) -> str:
        return (f"SequenceDataset("
                f"n_samples={self.n_samples}, "
                f"seq_len={self.seq_len}, "
                f"n_features={self.n_features}, "
                f"n_targets={self.n_targets})")

print("SequenceDataset defined.")

## Step 4: load_dataset()

Loads a single `(X, y)` pair of `.npy` files for a given **horizon** and **split**.

File naming convention (from Phase 1):
```
sequences_horizon_{horizon}_X_{split}.npy
sequences_horizon_{horizon}_y_{split}.npy
```

In [ ]:
def load_dataset(
    sequences_dir: str,
    horizon: int,
    split: str,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load pre-generated (X, y) NumPy files for one horizon and data split.

    Parameters
    ----------
    sequences_dir : str
        Path to the directory containing .npy files.
        In Colab: '/content/processed_data/sequences'
    horizon : int
        Prediction horizon, 1 to 10.
        horizon=1  -> predict 1 step ahead  (15 seconds)
        horizon=10 -> predict 10 steps ahead (2.5 minutes)
    split : str
        One of 'train', 'val', or 'test'.

    Returns
    -------
    X : np.ndarray  shape (n_samples, seq_len, n_features)
    y : np.ndarray  shape (n_samples, n_targets)

    Raises
    ------
    ValueError        if horizon not in [1,10] or split is unrecognised
    FileNotFoundError if .npy files are missing from sequences_dir
    """
    valid_splits = {'train', 'val', 'test'}
    if split not in valid_splits:
        raise ValueError(f"split must be one of {valid_splits}, got '{split}'")
    if not (1 <= horizon <= 10):
        raise ValueError(f"horizon must be 1-10, got {horizon}")

    base   = Path(sequences_dir)
    X_path = base / f"sequences_horizon_{horizon}_X_{split}.npy"
    y_path = base / f"sequences_horizon_{horizon}_y_{split}.npy"

    if not X_path.exists():
        raise FileNotFoundError(f"X file not found: {X_path}")
    if not y_path.exists():
        raise FileNotFoundError(f"y file not found: {y_path}")

    X = np.load(str(X_path))
    y = np.load(str(y_path))

    logger.info(f"[load_dataset] horizon={horizon} {split} | X={X.shape} y={y.shape}")
    return X, y

print("load_dataset() defined.")

## Step 5: create_dataloader()

Wraps `(X, y)` NumPy arrays in a PyTorch `DataLoader` with configurable settings.

Key choices:
- `shuffle=True` for **training only** — prevents the model from memorising batch order
- `num_workers=0` — safest setting in Colab (multiprocessing is restricted in notebooks)
- `pin_memory=True` when a GPU is available — speeds up CPU to GPU tensor transfer

In [ ]:
def create_dataloader(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int  = 64,
    shuffle: bool    = True,
    num_workers: int = 0,
    pin_memory: bool = False,
    drop_last: bool  = False,
) -> DataLoader:
    """
    Build a DataLoader from NumPy arrays.

    Parameters
    ----------
    X, y        : NumPy arrays (see load_dataset for shapes)
    batch_size  : Samples per gradient update. Typical: 32, 64, 128.
    shuffle     : True for train, False for val/test.
    num_workers : Parallel loading workers. Use 0 in Colab.
    pin_memory  : Set True when training on GPU (faster CPU->GPU copy).
    drop_last   : Discard final incomplete batch. Useful with BatchNorm layers.

    Returns
    -------
    DataLoader that yields (X_batch, y_batch) tensors each iteration.
    """
    dataset = SequenceDataset(X, y)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=drop_last,
    )

    logger.info(
        f"[create_dataloader] {len(dataset):,} samples | "
        f"{len(loader)} batches | batch_size={batch_size} | shuffle={shuffle}"
    )
    return loader

print("create_dataloader() defined.")

## Step 6: build_dataloaders()

Convenience function — builds all three DataLoaders (train / val / test) for a
given horizon in one call with the correct shuffle settings applied automatically.

In [ ]:
def build_dataloaders(
    sequences_dir: str,
    horizon: int,
    batch_size: int  = 64,
    num_workers: int = 0,
    pin_memory: bool = False,
) -> Dict[str, DataLoader]:
    """
    Load all three splits and return a dict of DataLoaders.

    Shuffle policy applied automatically:
        train -> shuffle=True
        val   -> shuffle=False
        test  -> shuffle=False

    Parameters
    ----------
    sequences_dir : str   Path to .npy files directory.
    horizon       : int   Prediction horizon (1-10).
    batch_size    : int   Shared batch size for all loaders.
    num_workers   : int   Parallel workers (0 for Colab).
    pin_memory    : bool  Pin tensors to CUDA memory when using GPU.

    Returns
    -------
    dict  {'train': DataLoader, 'val': DataLoader, 'test': DataLoader}

    Example
    -------
    loaders = build_dataloaders(str(sequences_path), horizon=1, batch_size=64)
    X_batch, y_batch = next(iter(loaders['train']))
    print(X_batch.shape)  # (64, 240, 27)
    print(y_batch.shape)  # (64, 4)
    """
    split_cfg = [
        ('train', True),
        ('val',   False),
        ('test',  False),
    ]

    loaders: Dict[str, DataLoader] = {}
    for split, shuffle in split_cfg:
        X, y = load_dataset(sequences_dir, horizon, split)
        loaders[split] = create_dataloader(
            X, y,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    return loaders

print("build_dataloaders() defined.")

## Step 7: initialize_weights()

Called via `model.apply(initialize_weights)` which visits **every sub-module**
in the model tree one at a time.

| Layer | Weight type | Method | Why |
|---|---|---|---|
| GRU | `weight_ih` (input to hidden) | Xavier uniform | Stable activation variance at input gate |
| GRU | `weight_hh` (hidden to hidden) | **Orthogonal** | Eigenvalues = 1, prevents gradient vanish/explode through time |
| GRU | biases | Zeros | Clean start |
| Linear | weights | Xavier uniform | Stable variance across FC layers |
| Linear | biases | Zeros | Clean start |

In [ ]:
def initialize_weights(module: nn.Module) -> None:
    """
    Best-practice weight initialisation for GRU + Linear layers.

    Called via:  model.apply(initialize_weights)
    PyTorch walks every sub-module and passes it here individually.

    Rules
    -----
    GRU weight_ih  -> Xavier uniform
        Keeps the variance of activations stable at the input gate.

    GRU weight_hh  -> Orthogonal initialisation
        Orthogonal matrices have unit-magnitude eigenvalues.
        Gradients neither explode nor vanish as they flow backward
        through the recurrent connections.
        This is the standard best practice for RNN weight init.

    GRU / Linear biases -> Zeros
        Neutral starting point; the model learns offsets from data.

    Linear weights -> Xavier uniform
        Keeps variance stable across the fully connected head layers.
    """
    if isinstance(module, nn.GRU):
        for name, param in module.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.zeros_(param.data)

    elif isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

print("initialize_weights() defined.")

## Step 8: GRUModel

```
Input  (batch, 240, 27)
    |
    v
GRU  (2 stacked layers, hidden=128, inter-layer dropout=0.2)
    |   only h_n[-1] is used — the top layer's final hidden state
    v   shape: (batch, 128)
Dropout (0.2)
    |
    v
Linear  128 -> 64
    |
    v
ReLU
    |
    v
Linear  64 -> 4
    |
    v
Output (batch, 4)
```

**Why `h_n[-1]` and not `gru_out[:, -1, :]`?**
Both are numerically identical. `h_n[-1]` is the explicit final hidden state
of the top GRU layer — the compressed temporal summary of the full 240-step window.

**Why no activation on the output layer?**
This is a regression task. Sigmoid/tanh would clip predictions.
MSELoss (Phase 3) expects raw unbounded values.

In [ ]:
class GRUModel(nn.Module):
    """
    Stacked GRU network for multi-step container resource forecasting.

    One independent instance is trained per prediction horizon.
    horizon=1  -> model predicts 1 timestep  (15 sec) ahead
    horizon=10 -> model predicts 10 timesteps (2.5 min) ahead

    Parameters
    ----------
    input_size  : int   Features per timestep. Default 27.
    hidden_size : int   GRU hidden units per layer. Default 128.
    num_layers  : int   Stacked GRU layers. Default 2.
    output_size : int   Regression targets. Default 4.
    dropout     : float Dropout probability applied between GRU layers
                        and before the FC head. Range [0, 1).
    """

    def __init__(
        self,
        input_size:  int   = 27,
        hidden_size: int   = 128,
        num_layers:  int   = 2,
        output_size: int   = 4,
        dropout:     float = 0.2,
    ) -> None:
        super().__init__()

        self.input_size  = input_size
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.output_size = output_size
        self.dropout_p   = dropout

        # ── GRU stack ────────────────────────────────────────────────────
        # batch_first=True  expects (batch, seq_len, features)
        #                   returns (batch, seq_len, hidden_size)
        # dropout is applied between stacked layers only
        # (PyTorch ignores dropout for single-layer GRUs)
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # ── Dropout before FC head ────────────────────────────────────────
        # Applied to the final hidden state.
        # Prevents co-adaptation of hidden units; reduces overfitting.
        self.dropout = nn.Dropout(p=dropout)

        # ── Fully connected head ──────────────────────────────────────────
        # Two-layer MLP compresses hidden_size -> 64 -> output_size.
        # The intermediate 64-unit layer learns non-linear combinations
        # of the GRU's compressed temporal representation.
        self.fc1  = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(64, output_size)

        # Apply weight initialisation to every sub-module in the model
        self.apply(initialize_weights)

        logger.info(
            f"[GRUModel] input={input_size} hidden={hidden_size} "
            f"layers={num_layers} output={output_size} dropout={dropout} | "
            f"params={self._count_params():,}"
        )

    # ── Forward pass ─────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : torch.Tensor  shape (batch_size, seq_len, input_size)
            A batch of input windows from the DataLoader.

        Returns
        -------
        torch.Tensor  shape (batch_size, output_size)
            Raw regression predictions (no output activation).

        Flow
        ----
        1. GRU processes the full 240-timestep sequence.
           gru_out : (batch, 240, hidden_size) -- all timestep outputs (unused)
           h_n     : (num_layers, batch, hidden_size) -- final hidden states

        2. h_n[-1] extracts the TOP layer's final hidden state: (batch, hidden_size)
           This is the most abstract temporal summary of the input window.

        3. Dropout -> FC1 -> ReLU -> FC2 maps it to 4 target predictions.
        """
        # Step 1: run GRU over the full sequence
        gru_out, h_n = self.gru(x)

        # Step 2: take only the top layer's final hidden state
        last_hidden = h_n[-1]            # (batch, hidden_size)

        # Step 3: FC head with dropout
        out = self.dropout(last_hidden)  # (batch, hidden_size)
        out = self.fc1(out)              # (batch, 64)
        out = self.relu(out)             # (batch, 64)
        out = self.fc2(out)              # (batch, output_size)

        return out

    # ── Utility methods ───────────────────────────────────────────────────

    def _count_params(self) -> int:
        """Return total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def summary(self) -> None:
        """Print a human-readable architecture summary."""
        sep = "=" * 58
        print(sep)
        print("GRUModel - Architecture Summary")
        print(sep)
        print(f"  Input size   : {self.input_size}  (features per timestep)")
        print(f"  Hidden size  : {self.hidden_size}")
        print(f"  GRU layers   : {self.num_layers}")
        print(f"  Dropout      : {self.dropout_p}")
        print(f"  Output size  : {self.output_size}  (regression targets)")
        print("-" * 58)
        print(f"  Layer           Output shape")
        print(f"  GRU (stack)     (batch, 240, {self.hidden_size}) -> h_n[-1]: (batch, {self.hidden_size})")
        print(f"  Dropout         (batch, {self.hidden_size})")
        print(f"  Linear  fc1     (batch, 64)")
        print(f"  ReLU            (batch, 64)")
        print(f"  Linear  fc2     (batch, {self.output_size})")
        print(sep)
        print(f"  Trainable parameters : {self._count_params():,}")
        print(sep)

    def __repr__(self) -> str:
        return (f"GRUModel(input={self.input_size}, hidden={self.hidden_size}, "
                f"layers={self.num_layers}, output={self.output_size}, "
                f"dropout={self.dropout_p}, params={self._count_params():,})")

print("GRUModel defined.")

## Step 9: Smoke Test

Verifies every component works correctly before connecting to real data.

**Checks:**
1. `GRUModel` forward pass produces correct output shape `(64, 4)`
2. `SequenceDataset` wraps arrays correctly and is indexable
3. `DataLoader` yields batches with the right shapes
4. `build_dataloaders()` loads real `.npy` files from the sequences directory

In [ ]:
print("=" * 58)
print("SMOKE TEST - Phase 2 Components")
print("=" * 58)

# ── Test 1: Model forward pass ────────────────────────────────
print("\n[1] GRUModel forward pass")
model = GRUModel(input_size=27, hidden_size=128, num_layers=2,
                 output_size=4, dropout=0.2)
model.summary()

dummy_input = torch.randn(64, 240, 27)       # (batch, seq_len, features)
dummy_out   = model(dummy_input)
print(f"    Input  shape : {tuple(dummy_input.shape)}")
print(f"    Output shape : {tuple(dummy_out.shape)}")
assert dummy_out.shape == (64, 4), "ERROR: wrong output shape"
print("    PASSED")

# ── Test 2: SequenceDataset ───────────────────────────────────
print("\n[2] SequenceDataset")
X_fake = np.random.randn(500, 240, 27).astype(np.float32)
y_fake = np.random.randn(500, 4).astype(np.float32)
ds = SequenceDataset(X_fake, y_fake)
print(f"    {ds}")
x0, y0 = ds[0]
assert x0.shape == (240, 27), f"ERROR: x shape {x0.shape}"
assert y0.shape == (4,),      f"ERROR: y shape {y0.shape}"
print("    PASSED")

# ── Test 3: DataLoader ────────────────────────────────────────
print("\n[3] create_dataloader")
loader = create_dataloader(X_fake, y_fake, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print(f"    Batch X : {tuple(xb.shape)}")
print(f"    Batch y : {tuple(yb.shape)}")
assert xb.shape == (64, 240, 27), f"ERROR: xb shape {xb.shape}"
assert yb.shape == (64, 4),       f"ERROR: yb shape {yb.shape}"
print("    PASSED")

# ── Test 4: Real .npy files ───────────────────────────────────
print("\n[4] build_dataloaders (real .npy files, horizon=1)")
if sequences_path.exists():
    try:
        loaders = build_dataloaders(
            str(sequences_path),
            horizon=1,
            batch_size=64,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )
        for split, ldr in loaders.items():
            xb, yb = next(iter(ldr))
            print(f"    {split:5s} -> X={tuple(xb.shape)}  y={tuple(yb.shape)}  batches={len(ldr)}")
        print("    PASSED")
    except FileNotFoundError as e:
        print(f"    SKIPPED - file not found: {e}")
else:
    print(f"    SKIPPED - path not found: {sequences_path}")

print("\n" + "=" * 58)
print("ALL SMOKE TESTS PASSED")
print("=" * 58)
print("\nPhase 2 complete. Waiting for approval to proceed to Phase 3.")